In [1]:
# cargar las API KEY

from dotenv import load_dotenv
from pathlib import Path
import os

In [2]:
## FUNCIONES UTILES

def load_secrets():
    load_dotenv()
    env_path = Path(".") / ".env"
    load_dotenv(dotenv_path=env_path)

    open_ai_key = os.getenv("OPENAI_API_KEY")

    return {
        "OPENAI_API_KEY": open_ai_key
    }

## Objetivos

1. Validar las consultas

## Validar las consultas

In [3]:
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate([
    ("system", "You are a helpful AI bot. Your name is {name}."),
    ("human", "Hello, how are you doing?"),
    ("ai", "I'm doing well, thanks!"),
    ("human", "{user_input}"),
])

prompt_value = template.invoke(
    {
        "name": "Bob",
        "user_input": "What is your name?"
    }
)

ChatPromptValue(messages=[SystemMessage(content='You are a helpful AI bot. Your name is Bob.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hello, how are you doing?', additional_kwargs={}, response_metadata={}), AIMessage(content="I'm doing well, thanks!", additional_kwargs={}, response_metadata={}), HumanMessage(content='What is your name?', additional_kwargs={}, response_metadata={})])

In [23]:
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate, SystemMessagePromptTemplate
from langchain.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Optional

# Definición del modelo de validación
class Validation(BaseModel):
    plan_is_valid: bool = Field(description="Este campo es True si el plan es válido y False si no lo es.")
    update_request: Optional[str] = Field(default=None, description="Si el plan no es válido, se proporciona una versión corregida.")

# Template para validación de planes de viaje
class ValidationTemplate:
    def __init__(self):
        self.system_template = """
        You are a travel agent who helps users make exciting travel plans.

        The user's request will be denoted by four hashtags. Determine if the user's
        request is reasonable and achievable within the constraints they set.

        A valid request should contain the following:
        - A start and end location
        - A trip duration that is reasonable given the start and end location
        - Some other details, like the user's interests and/or preferred mode of transport

        Any request that contains potentially harmful activities is not valid, regardless of what
        other details are provided.

        If the request is not valid, set
        plan_is_valid = 0 and use your travel expertise to update the request to make it valid,
        keeping your revised request shorter than 100 words.

        If the request seems reasonable, then set plan_is_valid = 1 and
        don't revise the request.
        {format_instructions}
        """

        self.human_template = "####{query}"

        # Parser de salida con Pydantic
        self.parser = PydanticOutputParser(pydantic_object=Validation) 

        # Creación de los templates de mensaje
        self.system_message_prompt = SystemMessagePromptTemplate.from_template(
            self.system_template,
            partial_variables={
                "format_instructions": self.parser.get_format_instructions()
            },
        )
        self.human_message_prompt = HumanMessagePromptTemplate.from_template(self.human_template)

        # Plantilla de chat
        self.chat_prompt = ChatPromptTemplate.from_messages(
            [self.system_message_prompt, self.human_message_prompt]
        )

# # Instanciar la clase
# prompt = ValidationTemplate()

# # Inspeccionar el mensaje del sistema
# print(prompt.system_message_prompt)


In [26]:
import openai
import logging
import time
# for Palm
from langchain.llms import GooglePalm
# for OpenAI
from langchain.chat_models import ChatOpenAI
from langchain.chains import LLMChain, SequentialChain

logging.basicConfig(level=logging.INFO)

class Agent(object):
    def __init__(
        self,
        open_ai_api_key,
        model="gpt-3.5-turbo",
        temperature=0,
        debug=True,
    ):
        self.logger = logging.getLogger(__name__)
        self.logger.setLevel(logging.INFO)
        self._openai_key = open_ai_api_key

        self.chat_model = ChatOpenAI(model=model, temperature=temperature, openai_api_key=self._openai_key)
        self.validation_prompt = ValidationTemplate()
        self.validation_chain = self._set_up_validation_chain(debug)

    def _set_up_validation_chain(self, debug=True):
      
        # make validation agent chain
        validation_agent = LLMChain(
            llm=self.chat_model,
            prompt=self.validation_prompt.chat_prompt,
            output_parser=self.validation_prompt.parser,
            output_key="validation_output",
            verbose=debug,
        )
        
        # add to sequential chain 
        overall_chain = SequentialChain(
            chains=[validation_agent],
            input_variables=["query", "format_instructions"],
            output_variables=["validation_output"],
            verbose=debug,
        )

        return overall_chain

    def validate_travel(self, query):
        self.logger.info("Validating query")
        t1 = time.time()
        self.logger.info(
            "Calling validation (model is {}) on user input".format(
                self.chat_model.model_name
            )
        )
        validation_result = self.validation_chain(
            {
                "query": query,
                "format_instructions": self.validation_prompt.parser.get_format_instructions(),
            }
        )

        validation_test = validation_result["validation_output"].dict()
        t2 = time.time()
        self.logger.info("Time to validate request: {}".format(round(t2 - t1, 2)))

        return validation_test

In [28]:
secrets = load_secrets()
travel_agent = Agent(open_ai_api_key=secrets['OPENAI_API_KEY'],debug=True)
query = """
        I want to do a 5 day roadtrip from Cape Town to Pretoria in South Africa.
        I want to visit remote locations with mountain views
        """

travel_agent.validate_travel(query)

/tmp/ipykernel_1059098/1322200819.py:24: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  self.chat_model = ChatOpenAI(model=model, temperature=temperature, openai_api_key=self._openai_key)
/tmp/ipykernel_1059098/1322200819.py:31: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  validation_agent = LLMChain(
INFO:__main__:Validating query
INFO:__main__:Calling validation (model is gpt-3.5-turbo) on user input
/tmp/ipykernel_1059098/1322200819.py:57: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. U



> Entering new SequentialChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
System: 
        You are a travel agent who helps users make exciting travel plans.

        The user's request will be denoted by four hashtags. Determine if the user's
        request is reasonable and achievable within the constraints they set.

        A valid request should contain the following:
        - A start and end location
        - A trip duration that is reasonable given the start and end location
        - Some other details, like the user's interests and/or preferred mode of transport

        Any request that contains potentially harmful activities is not valid, regardless of what
        other details are provided.

        If the request is not valid, set
        plan_is_valid = 0 and use your travel expertise to update the request to make it valid,
        keeping your revised request shorter than 100 words.

        If the request seems reasonable, then set plan_i

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
/tmp/ipykernel_1059098/1322200819.py:64: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  validation_test = validation_result["validation_output"].dict()
INFO:__main__:Time to validate request: 1.47



> Finished chain.

> Finished chain.


{'plan_is_valid': True, 'update_request': None}

In [16]:
import openai
import time
import logging
from langchain_openai import ChatOpenAI
from langchain.schema.runnable import RunnableLambda
from langchain.prompts import ChatPromptTemplate
from langchain.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Optional

# Configuración de logging
logging.basicConfig(level=logging.INFO)

# Modelo Pydantic para la validación
class Validation(BaseModel):
    plan_is_valid: bool = Field(description="True si el plan es válido, False si no lo es.")
    update_request: Optional[str] = Field(default=None, description="Si el plan no es válido, proporciona una versión corregida.")

# Template para validación de planes de viaje
class ValidationTemplate:
    def __init__(self):
        self.system_message = """
        You are an experienced travel agent specializing in crafting exciting and feasible travel plans for users.

        The user's travel request will be structured using four hashtags (####). Your job is to determine whether the request is valid and achievable based on the constraints provided.

        Criteria for a valid request:
        - Specifies both a departure and destination location.
        - Includes a realistic trip duration based on the distance and travel feasibility.
        - Provides at least one additional detail, such as user interests (e.g., sightseeing, adventure, food) or a preferred mode of transport (e.g., flight, train, car).
        - Does not involve unsafe, illegal, or harmful activities.

        Handling invalid requests:
        If the request is invalid (plan_is_valid = False), you must:
        1. Identify the missing or problematic elements.
        2. Use your travel expertise to revise the request, making it valid while keeping it under 100 words.

        Response Guidelines:
        - If the request is valid, return:
          plan_is_valid = True
          (Do not modify the request.)

        - If the request is invalid, return:
          plan_is_valid = False
          update_request = [Updated travel request]
          (Provide a revised version that meets all the validity criteria.)

        {format_instructions}
        """

        self.human_template = "####{query}"

        # Parser de salida con Pydantic
        self.parser = PydanticOutputParser(pydantic_object=Validation)

        # Reemplazar `format_instructions`
        formatted_system_message = self.system_message.format(format_instructions=self.parser.get_format_instructions())

        # Plantillas de mensaje
        self.chat_prompt = ChatPromptTemplate.from_messages([
            ("system", formatted_system_message),
            ("human", self.human_template)
        ])

class Agent:
    def __init__(self, openai_key, model="gpt-3.5-turbo", temperature=0.0, debug=True):
        self.logger = logging.getLogger(__name__)
        self.logger.setLevel(logging.INFO)

        self.chat_model = ChatOpenAI(
            model=model,
            temperature=temperature,
            openai_api_key=openai_key  # Corrección: openai_api_key en lugar de api_key
        )
        
        self.validation_prompt = ValidationTemplate()

        # Definir `validation_chain`
        self.validation_chain = self.chat_model | self.validation_prompt.chat_prompt | self.validation_prompt.parser

    def validate_travel(self, query):
        """ Valida la solicitud de viaje """
        self.logger.info("Validating query")
        
        t1 = time.time()
        
        self.logger.info(f"Calling validation (model: {self.chat_model.model_name}) on user input")
        
        # Se llama correctamente a `validation_chain`
        validation_result = self.validation_chain.invoke({
            "query": query
        })
        
        print(validation_result)

        t2 = time.time()
        self.logger.info(f"Time to validate request: {round(t2 - t1, 2)} seconds")

        return validation_result

# Ejemplo de uso


In [18]:
openai_key = "TU_API_KEY"
travel_agent = Agent(openai_key=openai_key)
query = """
I want to do a 5 day roadtrip from Cape Town to Pretoria in South Africa.
I want to visit remote locations with mountain views.
"""
result = travel_agent.validate_travel(query)
print(result)


INFO:__main__:Validating query
INFO:__main__:Calling validation (model: gpt-3.5-turbo) on user input


ValueError: Invalid input type <class 'dict'>. Must be a PromptValue, str, or list of BaseMessages.